# 01 - Bronze: Ingestão de Dados Brutos

Este notebook realiza a ingestão de dados brutos na camada Bronze:

- Lê arquivos CSV da pasta `/Workspace/Users/justinofilipe03@gmail.com/TechChallenge2/Bronze/source`
- Cria tabelas Delta gerenciadas em `workspace.bronze.*`
- Não aplica transformações complexas, apenas ingestão direta
- Preserva os dados originais para rastreabilidade

**Arquitetura Unity Catalog:**
- **Entrada**: Arquivos CSV no workspace
- **Saída**: Tabelas Delta em `workspace.bronze.*`

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("tech-challenge-bronze").getOrCreate()

BRONZE_CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SOURCE_PATH = "/Workspace/Users/justinofilipe03@gmail.com/TechChallenge2/Bronze/source"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"✓ Schema {BRONZE_CATALOG}.{BRONZE_SCHEMA} pronto")

✓ Schema workspace.bronze pronto


In [0]:
def normalizar_nome_coluna(nome):
    """Normalização MÍNIMA apenas para compatibilidade técnica Delta Lake
    (Delta não aceita espaços/acentos em nomes de colunas)
    Isso NÃO é tratamento de dados - é apenas conversão técnica de formato
    """
    import re
    import unicodedata
    
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('utf-8')
    nome = nome.lower()
    nome = re.sub(r'[^a-z0-9]+', '_', nome)
    nome = re.sub(r'_+', '_', nome).strip('_')
    return nome


def ingerir_csv_para_bronze(nome_arquivo, nome_tabela):
    """Lê CSV bruto e converte para Delta Lake
    
    Bronze = APENAS ingestão, SEM tratamento:
    - Lê CSV como está
    - Converte para Delta/Parquet
    - Mantém tipos originais (string)
    - Normaliza apenas nomes de colunas para compatibilidade técnica Delta
    
    TODO tratamento de dados (tipos, valores, validações) = Silver
    """
    csv_path = f"{SOURCE_PATH}/{nome_arquivo}"
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .option("encoding", "UTF-8") \
        .load(csv_path)
    
    # Normalizar nomes das colunas
    for col_name in df.columns:
        novo_nome = normalizar_nome_coluna(col_name)
        if novo_nome != col_name:
            df = df.withColumnRenamed(col_name, novo_nome)
    
    tabela_bronze = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{nome_tabela}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_bronze)
    
    count = df.count()
    print(f"✓ {nome_tabela}: {count:,} registros ingeridos")
    return df

## Ingestão das Tabelas Bronze

In [0]:
# Indicadores de alfabetização
indicador_uf = ingerir_csv_para_bronze("indicador_alfabetizacao_uf.csv", "indicador_alfabetizacao_uf")
indicador_municipio = ingerir_csv_para_bronze("indicador_alfabetizacao_municipio.csv", "indicador_alfabetizacao_municipio")

# Metas de alfabetização  
meta_brasil = ingerir_csv_para_bronze("meta_alfabetizacao_brasil.csv", "meta_alfabetizacao_brasil")
meta_uf = ingerir_csv_para_bronze("meta_alfabetizacao_uf.csv", "meta_alfabetizacao_uf")
meta_municipio = ingerir_csv_para_bronze("meta_alfabetizacao_municipio.csv", "meta_alfabetizacao_municipio")

# Microdados
microdados = ingerir_csv_para_bronze("microdados_alunos.csv", "microdados_alunos")

# Resultados e metas UFs 2025
resultados_uf_2025 = ingerir_csv_para_bronze("resultados_e_metas_ufs_2025.csv", "resultados_e_metas_ufs_2025")

✓ indicador_alfabetizacao_uf: 145 registros ingeridos
✓ indicador_alfabetizacao_municipio: 23,995 registros ingeridos
✓ meta_alfabetizacao_brasil: 3 registros ingeridos
✓ meta_alfabetizacao_uf: 54 registros ingeridos
✓ meta_alfabetizacao_municipio: 10,704 registros ingeridos
✓ microdados_alunos: 3,867,999 registros ingeridos
✓ resultados_e_metas_ufs_2025: 117 registros ingeridos


## Validação Final

In [0]:
tables = spark.sql(f"SHOW TABLES IN {BRONZE_CATALOG}.{BRONZE_SCHEMA}").collect()
print(f"\n=== Tabelas Bronze Criadas ({len(tables)}) ===")
for table in tables:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table.tableName}").count()
    print(f"  • {table.tableName}: {count:,} registros")


=== Tabelas Bronze Criadas (7) ===
  • indicador_alfabetizacao_municipio: 23,995 registros
  • indicador_alfabetizacao_uf: 145 registros
  • meta_alfabetizacao_brasil: 3 registros
  • meta_alfabetizacao_municipio: 10,704 registros
  • meta_alfabetizacao_uf: 54 registros
  • microdados_alunos: 3,867,999 registros
  • resultados_e_metas_ufs_2025: 117 registros
